# Streetview Detector Training in VS Code + Colab

Use this notebook only after connecting the notebook to a Colab kernel from VS Code.

Expected local files to upload to the Colab server:
- `USA_yolo_detection_with_clean_negatives.zip`
- `4.7_train_streetview_model.py`

Recommended VS Code flow:
1. Open this notebook in VS Code.
2. `Select Kernel` -> `Colab` -> `New Colab Server` and choose a GPU machine.
3. In VS Code Explorer, right-click the zip file and the training script, then choose `Upload to Colab`.
4. Run the cells below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, sys, platform, socket, shutil
import subprocess

print("python executable:", sys.executable)
print("platform:", platform.platform())
print("hostname:", socket.gethostname())
print("cwd:", os.getcwd())
print("which python:", shutil.which("python"))
print("which nvidia-smi:", shutil.which("nvidia-smi"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

print("\n=== OS ===")
subprocess.run(["bash", "-lc", "cat /etc/os-release || true"], check=False)

print("\n=== RAM ===")
subprocess.run(["bash", "-lc", "free -h || true"], check=False)

print("\n=== Disk ===")
subprocess.run(["bash", "-lc", "df -h /content || df -h / || true"], check=False)

print("\n=== PyTorch CUDA ===")
try:
    import torch
    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    print("cuda device count:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("gpu name:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch check failed:", e)


python executable: /usr/bin/python3
platform: Linux-6.6.113+-x86_64-with-glibc2.35
hostname: 87b4dbcae639
cwd: /content
which python: /usr/local/bin/python
which nvidia-smi: None
COLAB_GPU: 
CUDA_VISIBLE_DEVICES: None

=== OS ===

=== RAM ===

=== Disk ===

=== PyTorch CUDA ===
torch version: 2.10.0+cpu
cuda available: False
cuda device count: 0


In [2]:
!nvidia-smi
# !pip install -U ultralytics pyyaml

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
from pathlib import Path

USE_DRIVE_OUTPUT = False
DRIVE_RUNS_DIR = Path('/content/drive/MyDrive/ohio_taxation_colab/runs')

if USE_DRIVE_OUTPUT:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
    print('Using Drive-backed output dir:', DRIVE_RUNS_DIR)
else:
    print('Using ephemeral Colab output under /content/runs')


In [ ]:
from pathlib import Path
import shutil

CONTENT = Path('/content')
ZIP_NAME = 'USA_yolo_detection_with_clean_negatives.zip'
SCRIPT_NAME = '4.7_train_streetview_model.py'
DATASET_DIR_NAME = 'USA_yolo_detection_with_clean_negatives'

zip_matches = list(CONTENT.rglob(ZIP_NAME))
script_matches = list(CONTENT.rglob(SCRIPT_NAME))
dataset_matches = [p for p in CONTENT.rglob(DATASET_DIR_NAME) if p.is_dir()]

if dataset_matches:
    dataset_root = dataset_matches[0]
elif zip_matches:
    zip_path = zip_matches[0]
    work_dir = CONTENT / 'work'
    work_dir.mkdir(parents=True, exist_ok=True)
    dataset_root = work_dir / DATASET_DIR_NAME
    if dataset_root.exists():
        shutil.rmtree(dataset_root)
    shutil.unpack_archive(str(zip_path), str(work_dir))
else:
    raise FileNotFoundError(
        'Could not find uploaded dataset zip or extracted dataset folder under /content.'
    )

if not script_matches:
    raise FileNotFoundError(
        'Could not find uploaded 4.7_train_streetview_model.py under /content.'
    )

script_path = script_matches[0]
data_yaml = dataset_root / 'streetview_det_dataset.yaml'
manifest_path = dataset_root / 'manifests' / 'dataset_manifest.csv'

print('dataset_root =', dataset_root)
print('script_path  =', script_path)
print('data_yaml    =', data_yaml)
print('manifest     =', manifest_path)

assert data_yaml.exists(), data_yaml
assert manifest_path.exists(), manifest_path


In [ ]:
import shlex
import subprocess
from pathlib import Path

MODEL = 'yolo26l.pt'
RUN_NAME = 'streetview_det_yolo26l_vscode_v1'
PROJECT = str(DRIVE_RUNS_DIR if USE_DRIVE_OUTPUT else Path('/content/runs'))
IMGSZ = 768
BATCH = 8
EPOCHS = 120
WORKERS = 2
DEVICE = '0'

cmd = [
    'python', str(script_path),
    '--data', str(data_yaml),
    '--manifest', str(manifest_path),
    '--project', PROJECT,
    '--run_name', RUN_NAME,
    '--device', DEVICE,
    '--model', MODEL,
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--epochs', str(EPOCHS),
    '--workers', str(WORKERS),
]

print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)


In [ ]:
import shlex
import subprocess

resume_cmd = [
    'python', str(script_path),
    '--data', str(data_yaml),
    '--manifest', str(manifest_path),
    '--project', PROJECT,
    '--run_name', RUN_NAME,
    '--device', DEVICE,
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--workers', str(WORKERS),
    '--resume',
]

print(' '.join(shlex.quote(x) for x in resume_cmd))
# subprocess.run(resume_cmd, check=True)


In [ ]:
import shlex
import subprocess

eval_cmd = [
    'python', str(script_path),
    '--data', str(data_yaml),
    '--manifest', str(manifest_path),
    '--project', PROJECT,
    '--run_name', RUN_NAME,
    '--device', DEVICE,
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--eval_only',
]

print(' '.join(shlex.quote(x) for x in eval_cmd))
# subprocess.run(eval_cmd, check=True)


In [ ]:
import shutil
from pathlib import Path

run_dir = Path(PROJECT) / RUN_NAME
artifact_zip = Path('/content') / f'{RUN_NAME}_artifacts.zip'
if artifact_zip.exists():
    artifact_zip.unlink()
if run_dir.exists():
    shutil.make_archive(str(artifact_zip.with_suffix('')), 'zip', str(run_dir))
    print('artifact_zip =', artifact_zip)
else:
    print('Run directory does not exist yet:', run_dir)
